# Real Estate Feasibility Engine – Python App
## Step 9: One-click Excel report

**What this notebook adds**
* `engine/export.py` – builds a complete **Excel report** (8 sheets) from whatever is in the app, and can **reopen** it as a project.
* Sidebar of the app: **📊 Build Excel report → ⬇️ Download**, plus **📂 Open** now accepts a report as well as a project file.
* 5 new automatic checks (total **170**).

**The report's sheets**
| Sheet | Contents |
|---|---|
| Summary | status, 12 KPIs with benchmark colours, takeaways, P&L, cost-breakdown chart (prints on one landscape page) |
| Inputs | every input used – project, land, scenarios, zones, products with final timing, 21 cost heads with amounts, calculators, all payment plans, tax & benchmarks |
| P&L & Metrics | revenue statement, cost statement, P&L, JV split, all 18 key metrics with status |
| Cash Flow | quarter-by-quarter inflows, every cost head, landowner payments, project & developer cash flow, funding curve, FY summary + chart |
| Collections | bookings, booking-linked and construction-linked cash, receivables per product, milestone schedule |
| Timeline | coloured Gantt grid and the plain-English timing notes |
| Stress Test | scenario comparison, 4 one-way tables, 3 heat-map grids, headroom, tornado |
| Monte Carlo | probabilities, P10/P50/P90, probability curve + chart, and every single run |

Numbers are **values, not formulas**, so anyone can open, print or forward the file without the app.
Run cells top to bottom with **Shift + Enter**. Steps 1–8 must have been run once.

### Cell 1 – Open the project folder

In [ ]:
import os, subprocess, sys, time, urllib.request
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/feasibility_app'
except ImportError:
    BASE = os.path.abspath('feasibility_app')
os.chdir(BASE)
assert os.path.exists('stress_view.py'), 'Please run Step 8 first.'
try:
    import streamlit
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'])
print('Ready in', BASE)

### Cell 2 – The report builder (`engine/export.py`)

In [ ]:
%%writefile engine/export.py
"""
engine/export.py  -  STEP 9: one-click EXCEL REPORT of everything the app calculates.

build_excel_report(project, res, stress=None, mc=None) -> bytes of an .xlsx file with sheets:
    Summary · Inputs · P&L & Metrics · Cash Flow · Collections · Timeline · Stress Test · Monte Carlo
The numbers are values (not formulas) so anyone can open, read, print or forward the file without the app.
A hidden sheet keeps the full project, so the report can be opened again in the app (project_from_excel).
"""
from __future__ import annotations

import io
import json
from datetime import datetime

import numpy as np
from openpyxl import Workbook, load_workbook
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter as GL

from .inputs import CRORE, Project, from_dict, to_dict
from .report import key_metrics_table, kpi_cards, takeaways

NAVY, F = "1F3864", "Arial"
HEAD = PatternFill("solid", fgColor="D9E1F2"); SEC = PatternFill("solid", fgColor=NAVY); TOT = PatternFill("solid", fgColor="F2F2F2")
GOOD = PatternFill("solid", fgColor="E2F0D9"); REVIEW = PatternFill("solid", fgColor="FFF2CC")
thin = Side(style="thin", color="BFBFBF"); BOX = Border(left=thin, right=thin, top=thin, bottom=thin)
CR, PCT, INT, QF = '#,##0.00;(#,##0.00);"-"', '0.0%;(0.0%);"-"', '#,##0;(#,##0);"-"', '"Q"0'
JSON_SHEET = "_project"


class _Sheet:
    """Tiny helper that writes rows top-down with consistent styling."""
    def __init__(self, wb, title, widths=(34,) + (14,) * 45):
        self.ws = wb.create_sheet(title)
        self.ws.sheet_view.showGridLines = False
        for i, w in enumerate(widths): self.ws.column_dimensions[GL(i + 1)].width = w
        self.r = 1

    def title(self, text, sub=""):
        c = self.ws.cell(self.r, 1, text); c.font = Font(name=F, size=16, bold=True, color=NAVY); self.r += 1
        if sub: self.ws.cell(self.r, 1, sub).font = Font(name=F, size=9, italic=True, color="7F7F7F"); self.r += 1
        self.r += 1

    def section(self, text, width=10):
        for col in range(1, width + 1): self.ws.cell(self.r, col).fill = SEC
        self.ws.cell(self.r, 1, text).font = Font(name=F, size=11, bold=True, color="FFFFFF"); self.r += 1

    def header(self, cols, start=1):
        for i, h in enumerate(cols):
            c = self.ws.cell(self.r, start + i, h); c.font = Font(name=F, bold=True, size=10); c.fill = HEAD; c.border = BOX
            c.alignment = Alignment(wrap_text=True, horizontal="center", vertical="center")
        self.r += 1

    def row(self, values, fmts=None, bold=False, fill=None, start=1):
        for i, v in enumerate(values):
            if isinstance(v, (np.floating, np.integer)): v = v.item()
            if isinstance(v, float) and np.isnan(v): v = None
            c = self.ws.cell(self.r, start + i, v); c.font = Font(name=F, size=10, bold=bold); c.border = BOX
            if fmts and i < len(fmts) and fmts[i]: c.number_format = fmts[i]
            if fill: c.fill = fill
        self.r += 1
        return self.r - 1

    def gap(self, n=1): self.r += n

    def note(self, text):
        self.ws.cell(self.r, 1, text).font = Font(name=F, size=9, italic=True, color="7F7F7F"); self.r += 1


def build_excel_report(project: Project, res: dict, stress: dict | None = None, mc: dict | None = None) -> bytes:
    tl, cv, co, cr, cf = res["timeline"], res["curves"], res["collections"], res["costs"], res["cashflow"]
    pnl, m, info = cf.pnl, cf.metrics, project.info
    stamp = datetime.now().strftime("%d %b %Y %H:%M")
    wb = Workbook(); wb.remove(wb.active)
    n_q = max(tl.last_q, 1)

    # ------------------------------------------------------------------ 1. Summary
    s = _Sheet(wb, "Summary", (30, 16, 46, 12, 30, 16, 46, 12))
    s.title(f"{info.name} — Feasibility report",
            f"{info.location} · {info.project_type} · {info.deal_type} deal · scenario: {info.scenario} · generated {stamp} · values, not formulas")
    status = "✔ ALL CHECKS PASSED" if not res["problems"] else f"✖ {len(res['problems'])} CHECK(S) FAILED – review before use"
    c = s.ws.cell(s.r, 1, status); c.font = Font(name=F, bold=True, size=12, color="006100" if not res["problems"] else "C00000"); s.r += 2
    s.section("KEY PERFORMANCE INDICATORS (green = beats benchmark, amber = review)", 8)
    s.header(["KPI", "Value", "Detail", "", "KPI", "Value", "Detail", ""])
    cards = kpi_cards(project, res)
    for i in range(0, 12, 2):
        a, b = cards[i], cards[i + 1]
        rr = s.row([a["label"], a["value"], a["sub"], "", b["label"], b["value"], b["sub"], ""])
        for col, card in ((2, a), (6, b)):
            s.ws.cell(rr, col).fill = {"good": GOOD, "review": REVIEW}.get(card["status"], PatternFill())
            s.ws.cell(rr, col).font = Font(name=F, bold=True, size=11)
    s.gap()
    s.section("KEY TAKEAWAYS", 8)
    for i, t in enumerate(takeaways(project, res), 1):
        s.ws.cell(s.r, 1, f"{i}. {t}").alignment = Alignment(wrap_text=True, vertical="top")
        s.ws.merge_cells(start_row=s.r, start_column=1, end_row=s.r, end_column=8); s.ws.row_dimensions[s.r].height = 30; s.r += 1
    s.gap()
    s.section("PROFIT & LOSS (₹ Crore)", 8)
    s.header(["", "₹ Crore", "₹ / sq ft", "% of revenue"])
    sa = cr.total_sa or 1
    for lab, v, b in [("Total revenue (GDV)", pnl["revenue"], True), ("Total cost", pnl["total_cost"], False), ("EBITDA", pnl["ebitda"], True),
                      (f"Tax @ {pnl['tax_rate']:.1%} ({info.entity})", pnl["tax"], False), ("PAT", pnl["pat"], True)]:
        s.row([lab, v, v * CRORE / sa, v / pnl["revenue"] if pnl["revenue"] else 0], [None, CR, INT, PCT], bold=b)
    if problems := res["problems"]:
        s.gap(); s.section("CHECKS THAT FAILED", 8)
        for p_ in problems: s.note(p_)
    # cost breakdown chart
    s.gap(); top = s.r
    groups = [("Land / landowner", cr.amounts["land"] + cr.amounts["sd"] + cr.amounts["brok"]), ("Construction", cr.construction_heads_total),
              ("Marketing", cr.amounts["mkt"]), ("Overheads", cr.amounts["ovh"]), ("Legal & BAR", cr.amounts["legal"] + cr.amounts["bar"])]
    groups.append(("Other", cr.total_cost - sum(v for _, v in groups)))
    s.header(["Cost group", "₹ Crore"])
    g0 = s.r
    for lab, v in groups: s.row([lab, v], [None, CR])
    ch = BarChart(); ch.type = "bar"; ch.title = "Cost breakdown (₹ Cr)"; ch.legend = None; ch.height = 7; ch.width = 16
    ch.add_data(Reference(s.ws, min_col=2, min_row=g0, max_row=s.r - 1)); ch.set_categories(Reference(s.ws, min_col=1, min_row=g0, max_row=s.r - 1))
    ch.x_axis.delete = False; ch.y_axis.delete = False
    s.ws.add_chart(ch, f"E{top}")
    s.ws.page_setup.orientation = "landscape"; s.ws.page_setup.fitToWidth = 1; s.ws.page_setup.fitToHeight = 0
    s.ws.sheet_properties.pageSetUpPr.fitToPage = True

    # ------------------------------------------------------------------ 2. Inputs
    s = _Sheet(wb, "Inputs", (34, 22, 16, 16, 14, 14, 14, 12, 12, 14, 24, 12, 12, 14, 12, 44))
    s.title("Inputs used for this report", "Exactly as entered in the app (percentages shown as %)")
    s.section("PROJECT & LAND", 6)
    L = project.land
    for lab, v, f in [("Project name", info.name, None), ("Location", info.location, None), ("Project type", info.project_type, None),
                      ("Project start (Q1)", info.start_date, "dd-mmm-yyyy"), ("Deal type", info.deal_type, None), ("Taxable entity", info.entity, None),
                      ("Active scenario", info.scenario, None), ("Land area (sq yd)", L.area_sqyd, INT), ("Market rate (₹/sq yd)", L.market_rate, INT),
                      ("DLC rate (₹/sq m)", L.dlc_rate_sqm, INT), ("Landowner share (JV)", L.lo_share, PCT), ("Stamp duty JV (% of DLC)", L.stamp_duty_jv, '0.00%'),
                      ("Purchase rate – outright (₹/sq yd)", L.purchase_rate, INT), ("Stamp duty – outright", L.stamp_duty_outright, '0.00%'),
                      ("Land payment curve / start / end", f"{L.payment_curve} Q{L.payment_start_q}–Q{L.payment_end_q}", None),
                      ("Catch-up lag (quarters)", project.catch_up_lag, '0')]:
        s.row([lab, v], [None, f])
    s.gap(); s.section("SCENARIOS & ZONES", 6)
    s.header(["Scenario", "Sales period ×", "Price change", "Cost change", "Delay (Q)"])
    for sc in project.scenarios: s.row([sc.name, sc.sales_period_mult, sc.price_change, sc.cost_change, sc.delay_q], [None, '0.00"x"', PCT, PCT, '0'])
    s.header(["Zone", "Share of land"])
    for z in project.zones: s.row([z.name, z.share], [None, PCT])
    s.gap(); s.section("PRODUCTS (with final timing)", 16)
    s.header(["Product", "Zone", "Category", "Included", "Area (sq ft)", "Rate ₹/sq ft", "Constr. ₹/sq ft", "Sales period", "Sales curve",
              "Constr. qtrs", "Payment plan", "Launch", "Sales end", "Constr. start", "Constr. end", "How timing was decided"])
    for p in project.products:
        t = tl.products.get(p.name)
        s.row([p.name, p.zone, p.category, "Yes" if p.include else "No", p.area_sqft, p.rate, p.construction_rate, p.sales_period_q, p.sales_curve,
               p.construction_duration_q, p.clp_plan, t.launch if t else None, t.sales_end if t else None, t.cons_start if t else None,
               t.cons_end if t else None, t.note if t else "not active"], [None, None, None, None, INT, INT, INT, '0', None, '0', None, QF, QF, QF, QF, None])
    s.gap(); s.section("COST HEADS", 10)
    s.header(["Cost head", "Input 1", "Input 2 / basis", "Applicable", "Phasing curve", "Start Q", "End Q", "Amount ₹ Cr", "₹ / sq ft", "% of GDV"])
    for c_ in project.costs:
        a = cr.amounts[c_.key]
        s.row([c_.name, c_.rate1, c_.rate2 if c_.rate2 is not None else c_.basis, "Yes" if c_.applicable else "No", c_.curve, c_.start_q, c_.end_q,
               a, a * CRORE / sa, a / cr.total_revenue if cr.total_revenue else 0], [None, '#,##0.####', None, None, None, '0', '0', CR, INT, '0.00%'])
    s.row(["TOTAL COST", None, None, None, None, None, None, cr.total_cost, cr.total_cost * CRORE / sa, cr.total_cost / cr.total_revenue],
          [None] * 7 + [CR, INT, '0.00%'], bold=True, fill=TOT)
    s.gap(); s.section("CALCULATORS", 6)
    b = cr.bar
    s.row(["BAR / premium FAR", f"{project.bar.authority} / {project.bar.land_use}: extra {b['extra_sqm']:,.0f} sq m × ₹{project.bar.reserved_price_sqm:,.0f} × {b['pct']:.0%}", None, None, None, None, None, b["cost"]],
          [None] * 7 + [CR])
    s.row(["AIF interest (if switched on)", f"{project.aif.rate:.0%} for {project.aif.tenure_years} yrs, {project.aif.compounding}", None, None, None, None, None, cr.aif["cost"]], [None] * 7 + [CR])
    s.header(["Initial investment item (JV)", "₹ Crore", "Payout quarter"])
    for k_, v in cr.initial_items.items(): s.row([k_, v, project.initial.payout_q.get(k_)], [None, CR, QF])
    s.row(["Initial investment used (" + info.deal_type + ")", cr.initial_used], [None, CR], bold=True, fill=TOT)
    s.gap(); s.section("PAYMENT PLANS (CLP)", 6)
    s.header(["Plan / milestone", "Trigger", "Value", "% of sale", "Used by"])
    for pl in project.plans:
        s.row([pl.name, "", "", sum(m_.pct for m_ in pl.milestones), ", ".join(p.name for p in project.products if p.clp_plan == pl.name)],
              [None, None, None, PCT], bold=True, fill=TOT)
        for m_ in pl.milestones:
            s.row(["   " + m_.name, m_.trigger, m_.value, m_.pct], [None, None, '0' if m_.trigger.startswith("Booking") else '0%', PCT])
    s.gap(); s.section("TAX, COST OF EQUITY & BENCHMARKS", 6)
    fz = project.finance
    for lab, v, f in [("Tax – Company", fz.tax_company, PCT), ("Tax – LLP", fz.tax_llp, PCT), ("Cost of equity", fz.cost_of_equity, PCT)] + \
                     [(f"Benchmark – {k_.replace('_', ' ')}", v, '0.00"x"' if k_ == "equity_multiple" else PCT) for k_, v in fz.benchmarks.items()]:
        s.row([lab, v], [None, f])
    s.ws.freeze_panes = "B4"

    # ------------------------------------------------------------------ 3. P&L & Metrics
    s = _Sheet(wb, "P&L & Metrics", (40, 16, 16, 16, 16))
    s.title("Revenue, cost, P&L and key metrics")
    s.section("A. REVENUE STATEMENT", 5); s.header(["Product", "₹ Crore", "₹ / sq ft", "% of total", "Category"])
    sc_ = project.scenario()
    for p in project.active_products():
        s.row([p.name, cv.revenue[p.name], p.rate * (1 + sc_.price_change), cv.revenue[p.name] / cr.total_revenue, p.category], [None, CR, INT, PCT])
    s.row(["TOTAL REVENUE", cr.total_revenue, cr.total_revenue * CRORE / sa, 1.0], [None, CR, INT, PCT], bold=True, fill=TOT)
    s.gap(); s.section("B. COST STATEMENT", 5); s.header(["Cost head", "₹ Crore", "₹ / sq ft", "% of revenue", "Applicable"])
    for c_ in project.costs:
        a = cr.amounts[c_.key]
        s.row([c_.name, a, a * CRORE / sa, a / cr.total_revenue if cr.total_revenue else 0, "Yes" if c_.applicable else "No"], [None, CR, INT, '0.00%'])
    s.row(["TOTAL COST", cr.total_cost, cr.total_cost * CRORE / sa, cr.total_cost / cr.total_revenue], [None, CR, INT, PCT], bold=True, fill=TOT)
    s.gap(); s.section("C. PROFIT & LOSS", 5); s.header(["", "₹ Crore", "₹ / sq ft", "% of revenue"])
    for lab, v in [("Revenue", pnl["revenue"]), ("Total cost", pnl["total_cost"]), ("EBITDA", pnl["ebitda"]), ("Tax", pnl["tax"]), ("PAT", pnl["pat"]),
                   ("Developer's revenue share", pnl["developer_revenue"]), ("Developer's cost (excl. landowner share)", pnl["developer_cost"])]:
        s.row([lab, v, v * CRORE / sa, v / pnl["revenue"] if pnl["revenue"] else 0], [None, CR, INT, PCT], bold=lab in ("EBITDA", "PAT"))
    s.row(["EBITDA margin on developer's share", pnl["ebitda_margin_on_dev_share"]], [None, PCT])
    s.gap(); s.section("D. JV REVENUE SPLIT", 5); s.header(["Party", "Share", "₹ Crore"])
    s.row(["Developer", cr.dev_share, pnl["developer_revenue"]], [None, PCT, CR]); s.row(["Landowner", 1 - cr.dev_share, pnl["landowner_revenue"]], [None, PCT, CR])
    s.gap(); s.section("E. KEY INVESTMENT METRICS", 5); s.header(["Metric", "Value", "Benchmark", "Status"])
    for row_ in key_metrics_table(project, res):
        rr = s.row([row_["metric"], row_["value"], row_["benchmark"], row_["status"]])
        if row_["status"]: s.ws.cell(rr, 4).fill = GOOD if row_["status"] == "Strong" else REVIEW

    # ------------------------------------------------------------------ 4. Cash Flow
    s = _Sheet(wb, "Cash Flow", (44,) + (12,) * (n_q + 2))
    s.title("Quarterly cash flow (₹ Crore)", "Columns = quarters from the project start. Negative = cash out.")
    hdr = ["Quarter"] + [f"Q{q}" for q in range(1, n_q + 1)] + ["Total"]
    s.header(hdr)
    s.row(["Quarter end"] + tl.end_dates[:n_q], [None] + ["mmm-yy"] * n_q)
    s.row(["Financial year"] + tl.fy[:n_q])
    s.row(["Phase"] + tl.phase[:n_q])
    def qrow(label, arr, bold=False, fill=None, sign=1.0):
        vals = [sign * float(v) for v in arr[:n_q]]
        return s.row([label] + vals + [sum(vals)], [None] + [CR] * (n_q + 1), bold=bold, fill=fill)
    s.section("INFLOWS", n_q + 2)
    qrow("Customer collections", cf.inflow, True)
    qrow("   of which landowner share earned", co.landowner); qrow("   of which developer share", co.developer)
    s.section("OUTFLOWS", n_q + 2)
    for c_ in project.costs:
        if abs(cf.outflows[c_.key]).sum() > 1e-9: qrow("   " + c_.name, cf.outflows[c_.key], sign=-1)
    qrow("Total costs paid", cf.total_outflow, True, TOT, -1)
    qrow("Advance to landowner", cf.lo_advance, sign=-1); qrow("Cash paid to landowner (incl. advance)", cf.lo_paid, sign=-1)
    s.section("NET CASH FLOW", n_q + 2)
    qrow("Project net cash flow (before landowner)", cf.project_cf, True)
    r_dcf = qrow("DEVELOPER NET CASH FLOW (pre-tax)", cf.developer_cf, True, TOT)
    r_cum = s.row(["Developer cumulative (funding curve)"] + list(np.cumsum(cf.developer_cf)[:n_q]), [None] + [CR] * n_q, bold=True)
    qrow("Tax (indicative)", cf.tax, sign=-1); qrow("Developer net cash flow (post-tax, indicative)", cf.developer_cf_post_tax)
    s.gap(); s.section("YEARLY SUMMARY BY FINANCIAL YEAR", 6)
    s.header(["Financial year", "Collections", "Costs paid", "Paid to landowner", "Developer net", "Developer cumulative"])
    run = 0.0
    for fy, v in cf.fy_summary.items():
        if abs(v["collections"]) + abs(v["costs_paid"]) + abs(v["paid_to_landowner"]) < 1e-9: continue
        run += v["developer_net"]
        s.row([fy, v["collections"], v["costs_paid"], v["paid_to_landowner"], v["developer_net"], run], [None] + [CR] * 5)
    bar = BarChart(); bar.title = "Developer cash flow (₹ Cr) and funding curve"; bar.height = 8; bar.width = 28
    bar.add_data(Reference(s.ws, min_col=2, max_col=n_q + 1, min_row=r_dcf), from_rows=True, titles_from_data=False)
    bar.set_categories(Reference(s.ws, min_col=2, max_col=n_q + 1, min_row=4))
    ln = LineChart(); ln.add_data(Reference(s.ws, min_col=2, max_col=n_q + 1, min_row=r_cum), from_rows=True, titles_from_data=False)
    ln.y_axis.axId = 200; ln.y_axis.crosses = "max"; bar += ln
    for ax in (bar.x_axis, bar.y_axis, ln.y_axis): ax.delete = False
    bar.legend = None
    s.ws.add_chart(bar, f"A{s.r + 2}")
    s.ws.freeze_panes = "B5"

    # ------------------------------------------------------------------ 5. Collections
    s = _Sheet(wb, "Collections", (44,) + (12,) * (n_q + 2))
    s.title("Bookings, collections and payment-plan milestones (₹ Crore)")
    s.header(["Quarter"] + [f"Q{q}" for q in range(1, n_q + 1)] + ["Total"])
    for name in co.collections:
        s.section(name, n_q + 2)
        qrow("   Bookings (sales value)", co.bookings[name]); qrow("   Booking-linked cash", co.booking_linked[name])
        qrow("   Construction-linked cash", co.construction_linked[name]); qrow("   Collections", co.collections[name], True)
        s.row(["   Receivables (cumulative)"] + list(co.receivables[name][:n_q]), [None] + [CR] * n_q)
    s.section("PROJECT", n_q + 2)
    qrow("Total collections", co.total, True, TOT); qrow("Landowner share", co.landowner); qrow("Developer share", co.developer)
    s.gap(); s.section("MILESTONE SCHEDULE", 5); s.header(["Product", "Payment plan", "Milestone", "% of sale", "Falls due"])
    for x in co.schedule: s.row([x.product, x.plan, x.name, x.pct, x.text], [None, None, None, PCT])
    s.ws.freeze_panes = "B4"

    # ------------------------------------------------------------------ 6. Timeline
    s = _Sheet(wb, "Timeline", (34,) + (5,) * n_q)
    s.title("Project timeline", "S = selling · C = construction · # = both")
    s.header(["Product"] + [f"Q{q}" for q in range(1, n_q + 1)])
    for t in tl.products.values():
        vals = ["#" if (t.launch <= q <= t.sales_end and t.cons_start <= q <= t.cons_end) else "S" if t.launch <= q <= t.sales_end
                else "C" if t.cons_start <= q <= t.cons_end else "" for q in range(1, n_q + 1)]
        rr = s.row([t.name] + vals)
        for j, v in enumerate(vals):
            if v: s.ws.cell(rr, 2 + j).fill = PatternFill("solid", fgColor={"S": "A9D08E", "C": "9BC2E6", "#": "8EA9DB"}[v])
    s.row(["Phase"] + tl.phase[:n_q])
    s.gap()
    for t in tl.products.values(): s.note(f"{t.name}: {t.note}")

    # ------------------------------------------------------------------ 7. Stress Test
    s = _Sheet(wb, "Stress Test", (34, 16, 16, 16, 16, 16, 16, 16))
    s.title("Stress test", "Each line is a full model run with the lever shown (on top of the active scenario).")
    if stress:
        s.section("SCENARIO COMPARISON", 8)
        s.header(["Scenario", "GDV ₹Cr", "EBITDA ₹Cr", "PAT ₹Cr", "Developer IRR", "Dev NPV ₹Cr", "Peak ₹Cr", "Payback"])
        for x in stress["scenarios"]:
            s.row([x["scenario"], x["gdv"], x["ebitda"], x["pat"], x["developer_irr"], x["developer_npv"], x["peak"], f"Q{x['payback_q']}" if x["payback_q"] else "never"],
                  [None, CR, CR, CR, PCT, CR, CR])
        names = {"price": ("Sales price shock", '+0%;-0%;0%'), "cost": ("Construction cost shock", '+0%;-0%;0%'),
                 "speed": ("Sales period multiplier", '0.00"x"'), "delay": ("Extra delay (quarters)", '"+"0" Q"')}
        for lv, rows in stress["one_way"].items():
            s.gap(); s.section(f"ONE-WAY: {names[lv][0].upper()}", 8)
            s.header([names[lv][0], "Developer IRR", "Dev NPV ₹Cr", "PAT ₹Cr", "EBITDA ₹Cr", "Peak ₹Cr", "Payback"])
            for x in rows:
                s.row([x["value"], x["developer_irr"], x["developer_npv"], x["pat"], x["ebitda"], x["peak"], f"Q{x['payback_q']}" if x["payback_q"] else "never"],
                      [names[lv][1], PCT, CR, CR, CR, CR])
        for key, title, rows_, cols_, rf, cf_, vf in [
                ("irr_grid", "DEVELOPER IRR – construction cost (down) × price (across)", [0, .05, .1, .15, .2], [-.1, -.05, 0, .05, .1], '+0%;-0%;0%', '+0%;-0%;0%', PCT),
                ("pat_grid", "PAT ₹Cr – construction cost (down) × price (across)", [0, .05, .1, .15, .2], [-.1, -.05, 0, .05, .1], '+0%;-0%;0%', '+0%;-0%;0%', CR),
                ("peak_grid", "PEAK INVESTMENT ₹Cr – sales period (down) × delay (across)", [.75, 1, 1.25, 1.5, 2], [0, 1, 2, 4], '0.00"x"', '"+"0" Q"', CR)]:
            s.gap(); s.section(title, 8)
            s.row([""] + cols_, [None] + [cf_] * len(cols_), bold=True, fill=HEAD)
            for i, rv in enumerate(rows_): s.row([rv] + list(stress[key][i]), [rf] + [vf] * len(cols_))
        h = stress["headroom"]
        s.gap(); s.section("HEADROOM", 8)
        for lab, v in [("Price shock at which EBITDA = 0", h["price_to_zero_ebitda"]), ("Cost overrun at which EBITDA = 0", h["cost_to_zero_ebitda"]),
                       (f"Price shock at which developer IRR = {h['hurdle']:.0%}", h["price_to_hurdle"]),
                       (f"Cost overrun at which developer IRR = {h['hurdle']:.0%}", h["cost_to_hurdle"])]:
            s.row([lab, v if v is not None else "beyond tested range"], [None, '+0.0%;-0.0%'])
        s.gap(); s.section("WHAT MOVES DEVELOPER IRR THE MOST", 8); s.header(["Lever", "Downside case", "IRR", "Upside case", "IRR", "Change (downside)"])
        for x in stress["tornado"]:
            s.row([x["lever"], x["down_value"], x["down"], x["up_value"], x["up"], x["impact"]], [None, '0.00', PCT, '0.00', PCT, '+0.0%;-0.0%'])
    else:
        s.note("Stress test not included in this report.")

    # ------------------------------------------------------------------ 8. Monte Carlo
    s = _Sheet(wb, "Monte Carlo", (34, 16, 16, 16, 16, 16, 16, 16))
    s.title("Monte Carlo simulation")
    if mc:
        s.section("RESULTS", 6)
        for lab, v, f in [("Number of runs", mc["n"], INT), ("Random seed", mc["seed"], '0'), ("Runs that broke a model check", mc["failed_runs"], '0'),
                          (f"P(developer IRR ≥ {mc['hurdle']:.0%})", mc["p_irr_above_hurdle"], PCT), ("P(developer NPV > 0)", mc["p_npv_positive"], PCT),
                          ("P(loss: PAT < 0)", mc["p_loss"], PCT)]:
            s.row([lab, v], [None, f], bold=lab.startswith("P("))
        s.header(["", "P10 (pessimistic)", "P50 (median)", "P90 (optimistic)"])
        s.row(["Developer IRR", mc["irr_p10"], mc["irr_p50"], mc["irr_p90"]], [None, PCT, PCT, PCT])
        s.row(["Developer NPV ₹Cr", mc["npv_p10"], mc["npv_p50"], mc["npv_p90"]], [None, CR, CR, CR])
        s.row(["PAT ₹Cr", mc["pat_p10"], mc["pat_p50"], mc["pat_p90"]], [None, CR, CR, CR])
        s.row(["Peak investment ₹Cr (P50 / P90 / worst)", mc["peak_p50"], mc["peak_p90"], mc["peak_max"]], [None, CR, CR, CR])
        s.gap(); s.section("PROBABILITY THAT DEVELOPER IRR IS AT LEAST …", 6)
        irr_ok = mc["irr"][~np.isnan(mc["irr"])]
        s.header(["IRR at least", "Probability"])
        p0 = s.r
        for thr in [0.0, 0.10, 0.18, 0.25, 0.40, 0.60, 0.80, 1.00, 1.25]:
            s.row([thr, float(np.mean(irr_ok >= thr)) if len(irr_ok) else 0.0], [PCT, PCT])
        ch = LineChart(); ch.title = "Probability IRR ≥ X"; ch.height = 7; ch.width = 14; ch.legend = None
        ch.add_data(Reference(s.ws, min_col=2, min_row=p0, max_row=s.r - 1)); ch.set_categories(Reference(s.ws, min_col=1, min_row=p0, max_row=s.r - 1))
        ch.x_axis.delete = False; ch.y_axis.delete = False
        s.ws.add_chart(ch, f"D{p0 - 1}")
        s.gap(4); s.section("EVERY RUN (inputs drawn and results)", 8)
        s.header(["Run", "Price shock", "Cost shock", "Sales period ×", "Delay (Q)", "Developer IRR", "Dev NPV ₹Cr", "Peak ₹Cr"])
        sm = mc["samples"]
        for i in range(mc["n"]):
            s.row([i + 1, sm["price"][i], sm["cost"][i], sm["speed"][i], int(sm["delay"][i]), mc["irr"][i], mc["npv"][i], mc["peak"][i]],
                  ['0', PCT, PCT, '0.00', '0', PCT, CR, CR])
    else:
        s.note("Monte Carlo not run before this report was created (run it on the 'S · Stress & Monte Carlo' screen, then download again).")

    # ------------------------------------------------------------------ hidden project copy (to reopen in the app)
    ws = wb.create_sheet(JSON_SHEET)
    text = json.dumps(to_dict(project), ensure_ascii=False)
    for i in range(0, len(text), 30000):
        ws.cell(i // 30000 + 1, 1, text[i:i + 30000])
    ws.sheet_state = "hidden"

    wb.active = 0
    buf = io.BytesIO(); wb.save(buf)
    return buf.getvalue()


def project_from_excel(data: bytes) -> Project:
    """Re-open a report produced by build_excel_report in the app."""
    wb = load_workbook(io.BytesIO(data), read_only=True)
    ws = wb[JSON_SHEET]
    text = "".join(r[0] for r in ws.iter_rows(min_col=1, max_col=1, values_only=True) if r[0])
    return from_dict(json.loads(text))


### Cell 3 – Stress screen (small change: shares its cached tables with the report)

In [ ]:
%%writefile stress_view.py
"""
stress_view.py  -  STEP 8b: the 'Stress & Monte Carlo' screen.

Figure builders are plain functions (testable); render_stress() draws the page in Streamlit.
"""
from __future__ import annotations

import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from engine.inputs import from_dict
from engine.stress import (DEFAULT_STEPS, LEVERS, McRanges, headroom, monte_carlo, one_way, outcome, scenario_table,
                           tornado, two_way)

BLUE, GREEN, RED, ORANGE, NAVY = "#5B9BD5", "#70AD47", "#C00000", "#ED7D31", "#1F3864"
FMT = {"price": "{:+.0%}", "cost": "{:+.0%}", "speed": "{:.2f}x", "delay": "+{:d} Q"}


def _lay(fig, title, h=340):
    fig.update_layout(title=dict(text=title, x=0.01, font=dict(size=14)), height=h, margin=dict(l=10, r=10, t=45, b=10),
                      plot_bgcolor="white", legend=dict(orientation="h", y=-0.2))
    fig.update_yaxes(gridcolor="#EDEDED")
    return fig


def compute_tables(project) -> dict:
    """Everything that does not depend on the sliders (~150 model runs)."""
    return dict(
        scenarios=scenario_table(project),
        one_way={lv: one_way(project, lv) for lv in DEFAULT_STEPS},
        tornado=tornado(project),
        irr_grid=two_way(project, "cost", [0.0, 0.05, 0.10, 0.15, 0.20], "price", [-0.10, -0.05, 0.0, 0.05, 0.10], "developer_irr"),
        pat_grid=two_way(project, "cost", [0.0, 0.05, 0.10, 0.15, 0.20], "price", [-0.10, -0.05, 0.0, 0.05, 0.10], "pat"),
        peak_grid=two_way(project, "speed", [0.75, 1.0, 1.25, 1.5, 2.0], "delay", [0, 1, 2, 4], "peak"),
        headroom=headroom(project))


def fig_one_way(rows, lever, hurdle):
    x = [FMT[lever].format(int(r["value"]) if lever == "delay" else r["value"]) for r in rows]
    y = [r["developer_irr"] for r in rows]
    f = go.Figure(go.Scatter(x=x, y=y, mode="lines+markers+text", text=[f"{v:.0%}" if v is not None else "n/a" for v in y],
                             textposition="top center", line=dict(color=NAVY, width=3)))
    f.add_hline(y=hurdle, line_dash="dash", line_color=RED, annotation_text=f"hurdle {hurdle:.0%}")
    f.update_yaxes(tickformat=".0%")
    return _lay(f, LEVERS[lever], 280)


def fig_tornado(rows):
    labels = [r["lever"] + f"  ({FMT_KEY(r)})" for r in rows]
    f = go.Figure()
    f.add_bar(y=labels, x=[(r["down"] or 0) - r["base"] for r in rows], orientation="h", name="Downside", marker_color=RED,
              text=[f"{r['down']:.0%}" if r["down"] is not None else "n/a" for r in rows], textposition="outside")
    f.add_bar(y=labels, x=[(r["up"] or 0) - r["base"] for r in rows], orientation="h", name="Upside", marker_color=GREEN,
              text=[f"{r['up']:.0%}" if r["up"] is not None else "n/a" for r in rows], textposition="outside")
    f.update_layout(barmode="overlay"); f.update_xaxes(tickformat="+.0%", title="change in developer IRR vs base")
    return _lay(f, f"What moves developer IRR the most? (base {rows[0]['base']:.0%})", 320)


def FMT_KEY(r):
    lever = next(k for k, v in LEVERS.items() if v == r["lever"])
    show = lambda v: FMT[lever].format(int(v) if lever == "delay" else v)
    return f"{show(r['down_value'])} / {show(r['up_value'])}"


def fig_heat(grid, rows, cols, row_fmt, col_fmt, title, value_fmt, good_high=True, y_title="", x_title=""):
    f = px.imshow(grid, x=[col_fmt.format(c) for c in cols], y=[row_fmt.format(r) for r in rows], aspect="auto",
                  color_continuous_scale="RdYlGn" if good_high else "RdYlGn_r", text_auto=False)
    f.update_traces(text=[[value_fmt.format(v) for v in row] for row in grid], texttemplate="%{text}",
                    hovertemplate=f"{y_title} %{{y}} · {x_title} %{{x}}: %{{text}}<extra></extra>")
    f.update_layout(coloraxis_showscale=False); f.update_xaxes(title=x_title, side="top"); f.update_yaxes(title=y_title)
    return _lay(f, title, 320)


def fig_mc_hist(mc, threshold):
    irr = mc["irr"][~np.isnan(mc["irr"])]
    f = go.Figure(go.Histogram(x=irr, nbinsx=40, marker_color=BLUE, hovertemplate="IRR %{x:.0%}: %{y} runs<extra></extra>"))
    for v, lab, col in [(threshold, f"threshold {threshold:.0%}", RED), (mc["irr_p10"], "P10", ORANGE), (mc["irr_p50"], "P50", NAVY),
                        (mc["irr_p90"], "P90", GREEN)]:
        f.add_vline(x=v, line_dash="dash", line_color=col, annotation_text=lab, annotation_position="top")
    f.update_xaxes(tickformat=".0%", title="developer IRR"); f.update_yaxes(title="number of runs")
    return _lay(f, f"Developer IRR across {mc['n']:,} random scenarios", 360)


def fig_mc_exceedance(mc, threshold):
    irr = np.sort(mc["irr"][~np.isnan(mc["irr"])])
    prob = 1 - np.arange(len(irr)) / len(irr)
    f = go.Figure(go.Scatter(x=irr, y=prob, mode="lines", line=dict(color=NAVY, width=3),
                             hovertemplate="P(IRR ≥ %{x:.0%}) = %{y:.0%}<extra></extra>"))
    p_thr = float(np.mean(irr >= threshold)) if len(irr) else 0.0
    f.add_vline(x=threshold, line_dash="dash", line_color=RED)
    f.add_annotation(x=threshold, y=p_thr, text=f"{p_thr:.0%} chance ≥ {threshold:.0%}", showarrow=True, arrowhead=2)
    f.update_xaxes(tickformat=".0%", title="developer IRR at least …"); f.update_yaxes(tickformat=".0%", title="probability")
    return _lay(f, "Probability that developer IRR is at least X", 360)


def fig_mc_scatter(mc):
    f = go.Figure(go.Scatter(x=mc["peak"], y=mc["irr"], mode="markers", marker=dict(size=5, color=mc["samples"]["price"],
                  colorscale="RdYlGn", colorbar=dict(title="price<br>shock", tickformat=".0%"), opacity=0.7),
                  hovertemplate="peak ₹%{x:,.1f} Cr · IRR %{y:.0%}<extra></extra>"))
    f.update_xaxes(title="peak investment (₹ Cr)"); f.update_yaxes(tickformat=".0%", title="developer IRR")
    return _lay(f, "Return vs funding need for every run (colour = price shock)", 360)


def render_stress(P, fingerprint: str, tables_fn=None):
    """tables_fn(fingerprint) -> compute_tables result (the app passes a cached version)."""
    import streamlit as st
    cached = tables_fn or (lambda fp: compute_tables(from_dict(json.loads(fp))))

    st.header("Stress testing & Monte Carlo")
    st.caption("Levers act on top of the active scenario (page 1), exactly like the Excel STRESS sheet. Your inputs are never changed.")
    hurdle = P.finance.benchmarks["developer_irr"]

    # ---------- A. live sliders
    with st.container(border=True):
        st.subheader("A · Live what-if")
        c = st.columns(4)
        price = c[0].slider("Sales price shock", -30, 30, 0, 1, format="%d%%") / 100
        cost = c[1].slider("Construction cost shock", -10, 40, 0, 1, format="%d%%") / 100
        speed = c[2].slider("Sales period multiplier", 0.50, 2.50, 1.00, 0.05, format="%.2fx")
        delay = c[3].slider("Extra delay (quarters)", 0, 8, 0, 1)
        base, s = outcome(P), outcome(P, price=price, cost=cost, speed=speed, delay=delay)
        if not s["ok"]:
            st.warning("This combination breaks a model check (e.g. the project runs past 40 quarters): " + "; ".join(s["problems"][:2]))
        m = st.columns(5)
        pct = lambda v: "n/a" if v is None else f"{v:.1%}"
        dpp = lambda a, b: None if (a is None or b is None) else f"{(a - b) * 100:+.1f} pts"
        m[0].metric("Developer IRR", pct(s["developer_irr"]), dpp(s["developer_irr"], base["developer_irr"]))
        m[1].metric("Developer NPV", f"₹{s['developer_npv']:,.1f} Cr", f"{s['developer_npv'] - base['developer_npv']:+,.1f}")
        m[2].metric("PAT", f"₹{s['pat']:,.1f} Cr", f"{s['pat'] - base['pat']:+,.1f}")
        m[3].metric("Peak investment", f"₹{s['peak']:,.1f} Cr", f"{s['peak'] - base['peak']:+,.1f}", delta_color="inverse")
        m[4].metric("Payback", f"Q{s['payback_q']}" if s["payback_q"] else "never",
                    f"{(s['payback_q'] or 99) - (base['payback_q'] or 99):+d} qtrs" if s["payback_q"] and base["payback_q"] else None, delta_color="inverse")
        verdict = s["developer_irr"] is not None and s["developer_irr"] >= hurdle
        (st.success if verdict else st.error)(f"Developer IRR {'stays above' if verdict else 'falls below'} the {hurdle:.0%} hurdle in this case.")
        k = 32
        f = go.Figure()
        f.add_scatter(y=np.cumsum(base["developer_cf"])[:k], x=[f"Q{i}" for i in range(1, k + 1)], name="Base case", line=dict(color="#A5A5A5", width=3))
        f.add_scatter(y=np.cumsum(s["developer_cf"])[:k], x=[f"Q{i}" for i in range(1, k + 1)], name="Stressed", line=dict(color=RED, width=3))
        st.plotly_chart(_lay(f, "Developer cumulative cash (funding curve): base vs stressed (₹ Cr)", 320), width="stretch")

    T = cached(fingerprint)

    # ---------- B. scenarios + tornado
    c1, c2 = st.columns(2)
    with c1:
        st.subheader("B · Scenario comparison")
        df = pd.DataFrame(T["scenarios"])[["scenario", "gdv", "ebitda", "pat", "developer_irr", "developer_npv", "peak", "payback_q"]]
        df.columns = ["Scenario", "GDV ₹Cr", "EBITDA ₹Cr", "PAT ₹Cr", "Developer IRR", "Dev NPV ₹Cr", "Peak ₹Cr", "Payback"]
        df["Developer IRR"] = df["Developer IRR"].map(lambda v: "n/a" if v is None else f"{v:.1%}")
        df["Payback"] = df["Payback"].map(lambda q: f"Q{q}" if q else "never")
        st.dataframe(df.round(1), hide_index=True, width="stretch")
    with c2:
        st.plotly_chart(fig_tornado(T["tornado"]), width="stretch")

    # ---------- C. one-way
    st.subheader("C · One lever at a time – developer IRR")
    cols = st.columns(4)
    for col, lv in zip(cols, DEFAULT_STEPS):
        col.plotly_chart(fig_one_way(T["one_way"][lv], lv, hurdle), width="stretch")
    with st.expander("Show the one-way tables (IRR, NPV, PAT, EBITDA, peak, payback)"):
        for lv in DEFAULT_STEPS:
            d = pd.DataFrame(T["one_way"][lv])[["value", "developer_irr", "developer_npv", "pat", "ebitda", "peak", "payback_q"]]
            d.columns = [LEVERS[lv], "Developer IRR", "Dev NPV ₹Cr", "PAT ₹Cr", "EBITDA ₹Cr", "Peak ₹Cr", "Payback Q"]
            st.dataframe(d.round(4), hide_index=True, width="stretch")

    # ---------- D. heat maps
    st.subheader("D · Two levers together – heat maps")
    h = st.columns(3)
    pr, co_, sp, dl = [-0.10, -0.05, 0.0, 0.05, 0.10], [0.0, 0.05, 0.10, 0.15, 0.20], [0.75, 1.0, 1.25, 1.5, 2.0], [0, 1, 2, 4]
    h[0].plotly_chart(fig_heat(T["irr_grid"], co_, pr, "{:+.0%}", "{:+.0%}", "Developer IRR: price × cost", "{:.0%}",
                               y_title="cost", x_title="price"), width="stretch")
    h[1].plotly_chart(fig_heat(T["pat_grid"], co_, pr, "{:+.0%}", "{:+.0%}", "PAT (₹ Cr): price × cost", "{:,.0f}",
                               y_title="cost", x_title="price"), width="stretch")
    h[2].plotly_chart(fig_heat(T["peak_grid"], sp, dl, "{:.2f}x", "+{} Q", "Peak investment (₹ Cr): speed × delay", "{:,.1f}",
                               good_high=False, y_title="speed", x_title="delay"), width="stretch")

    # ---------- E. headroom
    st.subheader("E · Headroom – how much can go wrong?")
    hr = T["headroom"]
    pct0 = lambda v: "beyond the tested range" if v is None else f"{v:+.1%}"
    e = st.columns(4)
    e[0].metric("Price shock → EBITDA = 0", pct0(hr["price_to_zero_ebitda"]))
    e[1].metric("Cost overrun → EBITDA = 0", pct0(hr["cost_to_zero_ebitda"]))
    e[2].metric(f"Price shock → IRR = {hurdle:.0%}", pct0(hr["price_to_hurdle"]))
    e[3].metric(f"Cost overrun → IRR = {hurdle:.0%}", pct0(hr["cost_to_hurdle"]))
    st.caption(("Sales speed: IRR stays above the hurdle up to 3.0x slower. " if hr["speed_to_hurdle"] is None
                else f"Sales speed: IRR reaches the hurdle at {hr['speed_to_hurdle']:.2f}x. ") +
               ("Delay: IRR stays above the hurdle for every delay the model can test (up to 12 quarters)." if hr["delay_to_hurdle"] is None
                else f"Delay: IRR falls below the hurdle at +{hr['delay_to_hurdle']} quarters."))

    # ---------- F. Monte Carlo
    st.subheader("F · Monte Carlo – probabilities instead of single answers")
    st.caption("Each run picks a random price shock, cost shock, sales speed and delay from the ranges below "
               "(triangular: lowest · most likely · highest), runs the full model, and records the results.")
    with st.form("mc_form"):
        r = McRanges()
        c = st.columns(3)
        def tri(col, label, t, scale, fmt):
            lo = col.number_input(f"{label} – lowest", value=t[0] * scale, format=fmt)
            ml = col.number_input(f"{label} – most likely", value=t[1] * scale, format=fmt)
            hi = col.number_input(f"{label} – highest", value=t[2] * scale, format=fmt)
            return (lo / scale, ml / scale, hi / scale)
        price_r = tri(c[0], "Price shock %", r.price, 100, "%.1f")
        cost_r = tri(c[1], "Cost shock %", r.cost, 100, "%.1f")
        speed_r = tri(c[2], "Sales period ×", r.speed, 1, "%.2f")
        st.write("Probability of an extra delay")
        d = st.columns(4)
        probs = {q: d[i].number_input(f"+{q} quarter(s) %", value=p * 100, min_value=0.0, max_value=100.0, step=5.0) / 100
                 for i, (q, p) in enumerate(r.delay_probs.items())}
        c = st.columns(3)
        n = c[0].selectbox("Number of runs", [250, 500, 1000, 2000], index=2)
        seed = int(c[1].number_input("Random seed (same seed = same answer)", value=42, step=1))
        go_ = c[2].form_submit_button("▶ Run Monte Carlo", type="primary")
    bad = [lab for lab, t in [("price", price_r), ("cost", cost_r), ("speed", speed_r)] if not (t[0] <= t[1] <= t[2])]
    if bad:
        st.error(f"For {', '.join(bad)}: lowest ≤ most likely ≤ highest is required.")
    elif sum(probs.values()) <= 0:
        st.error("Delay probabilities must add up to more than 0%.")
    elif go_:
        bar = st.progress(0.0, text="Running…")
        res = monte_carlo(P, McRanges(price_r, cost_r, speed_r, probs), n=n, seed=seed, progress=lambda x: bar.progress(x, text=f"Running… {x:.0%}"))
        bar.empty()
        st.session_state.mc = dict(fp=fingerprint, res=res)
    mc_state = st.session_state.get("mc")
    if mc_state:
        mc = mc_state["res"]
        if mc_state["fp"] != fingerprint:
            st.warning("These results were calculated before your latest input changes – press ▶ Run Monte Carlo again.")
        thr = st.slider("Show the probability of developer IRR being at least…", 0, 150, int(round(mc["hurdle"] * 100)), 1, format="%d%%") / 100
        irr_ok = mc["irr"][~np.isnan(mc["irr"])]
        p_thr = float(np.mean(irr_ok >= thr)) if len(irr_ok) else 0.0
        k = st.columns(4)
        k[0].metric(f"P(developer IRR ≥ {thr:.0%})", f"{p_thr:.1%}")
        k[1].metric("P(NPV > 0)", f"{mc['p_npv_positive']:.1%}")
        k[2].metric("P(loss: PAT < 0)", f"{mc['p_loss']:.1%}")
        k[3].metric("Peak investment P90", f"₹{mc['peak_p90']:,.1f} Cr", f"worst ₹{mc['peak_max']:,.1f} Cr", delta_color="off")
        st.info(f"In {mc['n']:,} random scenarios the developer IRR is between **{mc['irr_p10']:.0%}** (P10 – only 1 in 10 runs is worse) "
                f"and **{mc['irr_p90']:.0%}** (P90), with a median of **{mc['irr_p50']:.0%}**. "
                f"There is a **{p_thr:.0%}** chance it is at least {thr:.0%}." +
                (f" ({mc['failed_runs']} runs broke a model check.)" if mc["failed_runs"] else ""), icon="🎲")
        c1, c2 = st.columns(2)
        c1.plotly_chart(fig_mc_hist(mc, thr), width="stretch")
        c2.plotly_chart(fig_mc_exceedance(mc, thr), width="stretch")
        st.plotly_chart(fig_mc_scatter(mc), width="stretch")
        st.dataframe(pd.DataFrame({"": ["P10 (pessimistic)", "P50 (median)", "P90 (optimistic)"],
                                   "Developer IRR": [f"{mc['irr_p10']:.1%}", f"{mc['irr_p50']:.1%}", f"{mc['irr_p90']:.1%}"],
                                   "Developer NPV ₹Cr": [round(mc["npv_p10"], 1), round(mc["npv_p50"], 1), round(mc["npv_p90"], 1)],
                                   "PAT ₹Cr": [round(mc["pat_p10"], 1), round(mc["pat_p50"], 1), round(mc["pat_p90"], 1)]}),
                     hide_index=True)


### Cell 4 – Updated app (`app.py`) – report buttons in the sidebar

In [ ]:
%%writefile app.py
"""
app.py  -  STEP 6: the input screens of the Real Estate Feasibility Engine (Streamlit).

Run with:   streamlit run app.py
Every screen edits ONE Project object kept in memory (st.session_state). After every change the whole
engine re-runs (a fraction of a second), and any problem is shown right next to the field / row that caused it.
"""
from __future__ import annotations

import copy
import json
from datetime import date

import pandas as pd
import streamlit as st

from engine.inputs import (
    BAR_AUTHORITIES, BAR_LAND_USES, BASIC_CURVES, CATEGORIES, COMPOUNDING, CONS_MODES, CONSTRUCTION_HEADS,
    CONTINGENCY_BASES, COST_CURVES, CRORE, DEAL_TYPES, EDC_BASES, ENTITIES, HORIZON_Q, LAND_CURVES, LAUNCH_MODES,
    LINK_EVENTS, PROJECT_TYPES, SPARE_BASES, SQFT_PER_SQYD, SQYD_PER_ACRE, SQYD_PER_BIGHA, TRIGGERS, ZONES,
    CostHead, ClpPlan, Milestone, Product, Scenario, Timing, check_inputs, from_dict, sample_project, to_dict,
)
from engine.cashflow import run_model
from dashboard import render_dashboard
from stress_view import compute_tables, render_stress
from engine.export import build_excel_report, project_from_excel

st.set_page_config(page_title="Real Estate Feasibility Engine", page_icon="🏗️", layout="wide")

# ============================================================================ state & engine run
if "project" not in st.session_state:
    st.session_state.project = sample_project()
P = st.session_state.project


def rerun_after_edit(editor_key: str):
    """Commit pattern for editable tables: the project is already updated -> clear the table's edit
    memory and redraw everything from the project."""
    st.session_state.pop(editor_key, None)
    st.rerun()


FINGERPRINT = json.dumps(to_dict(P), sort_keys=True, default=str)   # project state used for this run's results


@st.cache_data(show_spinner="Running the stress tables (about 150 model runs)…", max_entries=8)
def stress_tables(fp: str):
    """Stress tables for one project state – shared by the stress screen and the Excel report."""
    return compute_tables(from_dict(json.loads(fp)))
issues = check_inputs(P)
try:
    RES = run_model(P)
except Exception as exc:                      # an unforeseen input combination – never crash the screens
    RES, ENGINE_ERROR = None, str(exc)
else:
    ENGINE_ERROR = ""

PAGE_SECTIONS = {
    "0 · Dashboard": [],
    "S · Stress & Monte Carlo": [],
    "1 · Project & land": ["Project", "Land"],
    "2 · Scenarios & zones": ["Scenarios", "Zones"],
    "3 · Products": ["Products"],
    "4 · Product timing": ["Timing"],
    "5 · Cost heads": ["Costs"],
    "6 · Calculators": ["BAR", "AIF", "Initial investment"],
    "7 · Payment plans": ["CLP"],
    "8 · Tax & benchmarks": ["Finance"],
    "9 · Checks & summary": [],
}


def issues_for(section: str, item: str | None = None) -> list:
    return [i for i in issues if i.section == section and (item is None or i.item == item)]


def show(section: str, item: str):
    """Show the error(s) for one field directly under it."""
    for i in issues_for(section, item):
        st.error(i.message, icon="⚠️")


def row_status(section: str, item: str) -> str:
    msgs = [i.message.split(": ", 1)[-1] for i in issues_for(section, item)]
    return "✔" if not msgs else "⚠ " + " | ".join(msgs)


def pct_in(label, value, key=None, help=None, max_value=100.0, step=0.25):
    """Percentage field: shown as 32 (%) but stored as 0.32."""
    return st.number_input(label, value=round(float(value) * 100, 6), min_value=-100.0, max_value=max_value,
                           step=step, format="%.2f", key=key, help=help) / 100


# ============================================================================ sidebar
with st.sidebar:
    st.title("🏗️ Feasibility Engine")
    st.caption(f"{P.info.name} · {P.info.project_type} · {P.info.deal_type} · {P.info.scenario}")
    labels = []
    for page, secs in PAGE_SECTIONS.items():
        n = sum(1 for i in issues if i.section in secs)
        labels.append(f"{page}  ⚠️ {n}" if n else page)
    choice = st.radio("Go to", labels, label_visibility="collapsed")
    page = choice.split("  ⚠️")[0]

    st.divider()
    engine_problems = RES["problems"] if RES else []
    if not issues and RES and not engine_problems:
        st.success("✔ ALL CHECKS PASSED")
    else:
        n_problems = len(issues) + sum(1 for p in engine_problems if not any(p.endswith(i.message) for i in issues))
        st.error(f"✖ {n_problems or 1} issue(s) – see page 9")
    if RES:
        m, pnl = RES["cashflow"].metrics, RES["cashflow"].pnl
        c1, c2 = st.columns(2)
        c1.metric("GDV", f"₹{pnl['revenue']:,.0f} Cr")
        c2.metric("EBITDA", f"{pnl['ebitda_margin']:.1%}")
        c1.metric("Developer IRR", "n/a" if m["developer_irr"] is None else f"{m['developer_irr']:.1%}")
        c2.metric("Peak funding", f"₹{m['peak_investment']:,.1f} Cr")
        st.caption(f"Payback: {'Q' + str(m['payback_q']) if m['payback_q'] else 'never'} · "
                   f"PAT ₹{pnl['pat']:,.1f} Cr · {RES['timeline'].last_q} quarters")

    st.divider()
    st.markdown("**📊 Excel report**")
    inc_stress = st.checkbox("Include stress tests (+ ~2 s)", value=True)
    mc_now = st.session_state.get("mc")
    mc_ok = bool(mc_now and mc_now["fp"] == FINGERPRINT)
    st.caption("Monte Carlo: " + ("included (latest run)" if mc_ok else "not included – run it on 'S · Stress & Monte Carlo' first"))
    if st.button("Build Excel report", width="stretch", disabled=RES is None):
        with st.spinner("Building the report…"):
            data = build_excel_report(P, RES, stress_tables(FINGERPRINT) if (inc_stress and not RES["problems"]) else None,
                                      mc_now["res"] if mc_ok else None)
        st.session_state.report = dict(fp=FINGERPRINT, data=data, when=date.today().strftime("%d %b %Y"))
    rep = st.session_state.get("report")
    if rep and rep["fp"] == FINGERPRINT:
        st.download_button("⬇️ Download Excel report", rep["data"], type="primary", width="stretch",
                           file_name=f"{P.info.name.split(' (')[0].replace(' ', '_')}_Feasibility_Report.xlsx",
                           mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet")
    elif rep:
        st.caption("Inputs changed since the last report – build it again.")
    st.divider()
    st.download_button("💾 Save project file", json.dumps(to_dict(P), indent=2, ensure_ascii=False),
                       file_name=f"{P.info.name.split(' (')[0].replace(' ', '_')}.json", mime="application/json",
                       width="stretch")
    up = st.file_uploader("📂 Open project file or Excel report", type=["json", "xlsx"])
    if up is not None and st.session_state.get("loaded_file") != up.file_id:
        try:
            raw = up.getvalue()
            st.session_state.project = project_from_excel(raw) if up.name.lower().endswith(".xlsx") else from_dict(json.loads(raw))
            st.session_state.loaded_file = up.file_id
            st.rerun()
        except Exception as exc:
            st.error(f"Could not open this file: {exc}")
    if st.button("↺ Reset to sample project", width="stretch"):
        st.session_state.project = sample_project()
        st.rerun()

# ============================================================================ page 0: dashboard
if page.startswith("0"):
    if RES:
        render_dashboard(P, RES)
    else:
        st.error("The dashboard cannot be drawn until the errors on the other pages are fixed (see '9 · Checks & summary').")

# ============================================================================ stress testing & Monte Carlo
elif page.startswith("S"):
    if RES and not RES["problems"]:
        render_stress(P, FINGERPRINT, stress_tables)
    else:
        st.error("Stress tests need a model without errors – fix the issues listed on '9 · Checks & summary' first.")

# ============================================================================ page 1: project & land
elif page.startswith("1"):
    st.header("1 · Project & land")
    st.caption("Fields with ⚠️ underneath need fixing. All amounts in ₹; land in sq yd.")
    a, b = st.columns(2)
    with a:
        st.subheader("Project")
        P.info.name = st.text_input("Project name", P.info.name)
        P.info.location = st.text_input("Location", P.info.location)
        P.info.project_type = st.selectbox("Project type", PROJECT_TYPES, PROJECT_TYPES.index(P.info.project_type),
                                           help="Only products whose zone belongs to this type are calculated (Mixed = all).")
        P.info.start_date = st.date_input("Project start = Q1 (quarter-end date)", P.info.start_date,
                                          min_value=date(2020, 1, 1), max_value=date(2045, 12, 31), format="DD/MM/YYYY")
        show("Project", "start_date")
        P.info.deal_type = st.radio("Deal type", DEAL_TYPES, DEAL_TYPES.index(P.info.deal_type), horizontal=True)
        P.info.entity = st.radio("Taxable entity", ENTITIES, ENTITIES.index(P.info.entity), horizontal=True)
        names = [s.name for s in P.scenarios]
        P.info.scenario = st.selectbox("Active scenario", names, names.index(P.info.scenario) if P.info.scenario in names else 0)
        show("Project", "scenario")
        ov = st.number_input("Lifecycle for ROCE / CAGR (years) – 0 = use project span", value=float(P.info.lifecycle_override_years or 0),
                             min_value=0.0, step=0.5)
        P.info.lifecycle_override_years = ov or None
    with b:
        st.subheader("Land")
        L = P.land
        L.area_sqyd = st.number_input("Land area (sq yd)", value=float(L.area_sqyd), min_value=0.0, step=100.0)
        st.caption(f"= {L.area_sqyd * SQFT_PER_SQYD:,.0f} sq ft · {L.area_sqyd / SQYD_PER_ACRE:,.2f} acres · "
                   f"{L.area_sqyd / SQYD_PER_BIGHA:,.2f} bigha")
        L.market_rate = st.number_input("Current market rate (₹ / sq yd)", value=float(L.market_rate), min_value=0.0, step=500.0)
        L.dlc_rate_sqm = st.number_input("DLC / circle rate (₹ / sq m)", value=float(L.dlc_rate_sqm), min_value=0.0, step=100.0)
        show("Land", "land_rates")
        if P.info.deal_type == "JV":
            L.lo_share = pct_in("Landowner revenue share (%)", L.lo_share)
            show("Land", "lo_share")
            L.stamp_duty_jv = pct_in("Stamp duty + registration on JV (% of DLC value)", L.stamp_duty_jv, step=0.05)
            if RES:
                cr = RES["costs"]
                lo_px = cr.land_cost * CRORE / L.area_sqyd if L.area_sqyd else 0
                st.info(f"Landowner effectively gets ₹{lo_px:,.0f} / sq yd = "
                        f"{lo_px / L.market_rate - 1:+.1%} vs today's market rate.", icon="💡")
        else:
            L.purchase_rate = st.number_input("Purchase rate (₹ / sq yd)", value=float(L.purchase_rate), min_value=0.0, step=500.0)
            show("Land", "purchase_rate")
            L.stamp_duty_outright = pct_in("Stamp duty + registration – outright (%)", L.stamp_duty_outright, step=0.05)
            c1, c2, c3 = st.columns(3)
            L.payment_curve = c1.selectbox("Land payment curve", LAND_CURVES, LAND_CURVES.index(L.payment_curve))
            L.payment_start_q = int(c2.number_input("Start quarter", value=int(L.payment_start_q), min_value=1, max_value=HORIZON_Q))
            L.payment_end_q = int(c3.number_input("End quarter", value=int(L.payment_end_q), min_value=1, max_value=HORIZON_Q,
                                                  disabled=L.payment_curve == "Upfront"))
            show("Land", "payment")

# ============================================================================ page 2: scenarios & zones
elif page.startswith("2"):
    st.header("2 · Scenarios & development zones")
    st.subheader("Scenarios")
    st.caption("Sales period multiplier 1.5 = products take 50% longer to sell. Delay pushes every fixed launch / construction start later.")
    df = pd.DataFrame([{"Scenario": s.name, "Sales period ×": s.sales_period_mult, "Price change %": s.price_change * 100,
                        "Construction cost change %": s.cost_change * 100, "Delay (quarters)": s.delay_q,
                        "Check": row_status("Scenarios", s.name)} for s in P.scenarios])
    ed = st.data_editor(df, key="ed_scen", hide_index=True, width="stretch", num_rows="dynamic",
                        column_config={"Sales period ×": st.column_config.NumberColumn(min_value=0.1, step=0.05, format="%.2f"),
                                       "Delay (quarters)": st.column_config.NumberColumn(min_value=0, step=1),
                                       "Check": st.column_config.TextColumn(disabled=True)})
    if not ed.equals(df):
        P.scenarios = [Scenario(str(r["Scenario"] or f"Scenario {i+1}"), float(r["Sales period ×"] or 1), float(r["Price change %"] or 0) / 100,
                                float(r["Construction cost change %"] or 0) / 100, int(r["Delay (quarters)"] or 0))
                       for i, r in ed.iterrows()]
        rerun_after_edit("ed_scen")
    show("Scenarios", "names")

    st.subheader("Development zones")
    used = {"Villa": ["Villa Development"], "Apartments": ["Apartment Development"], "Plotted": ["Plotted Development"]}.get(P.info.project_type, ZONES)
    df = pd.DataFrame([{"Zone": z.name, "Share of land %": z.share * 100,
                        "Land area (sq ft)": round(z.share * P.land.area_sqyd * SQFT_PER_SQYD),
                        "In this project type?": "Yes" if (z.name in used or z.name == "Other / Clubhouse") else "No"} for z in P.zones])
    ed = st.data_editor(df, key="ed_zone", hide_index=True, width="stretch", num_rows="fixed",
                        column_config={"Zone": st.column_config.TextColumn(disabled=True),
                                       "Land area (sq ft)": st.column_config.NumberColumn(disabled=True, format="%d"),
                                       "In this project type?": st.column_config.TextColumn(disabled=True)})
    if not ed.equals(df):
        for z, (_, r) in zip(P.zones, ed.iterrows()):
            z.share = float(r["Share of land %"] or 0) / 100
        rerun_after_edit("ed_zone")
    st.caption(f"Total share: {sum(z.share for z in P.zones):.1%}")
    show("Zones", "zones")

# ============================================================================ page 3: products
elif page.startswith("3"):
    st.header("3 · Product mix")
    st.caption("One row per product. Add a row with the ➕ at the bottom of the table; tick **Include** to use it. "
               "**Active** is automatic: included AND its zone fits the project type.")
    active = {p.name for p in P.active_products()}
    plans = [pl.name for pl in P.plans]
    rev = RES["curves"].revenue if RES else {}
    df = pd.DataFrame([{
        "Include": p.include, "Product": p.name, "Zone": p.zone, "Category": p.category,
        "Area (sq ft)": p.area_sqft, "Sales rate (₹/sq ft)": p.rate, "Construction (₹/sq ft)": p.construction_rate,
        "Launch Q (fixed)": p.launch_q, "Sales period (qtrs)": p.sales_period_q, "Sales curve": p.sales_curve,
        "Payment plan": p.clp_plan, "Construction start Q (fixed)": p.construction_start_q,
        "Construction (qtrs)": p.construction_duration_q, "Construction curve": p.construction_curve,
        "Active": "Yes" if p.name in active else ("Excluded by type" if p.include else "Off"),
        "Revenue (₹ Cr)": round(rev.get(p.name, 0.0), 2), "Check": row_status("Products", p.name)} for p in P.products])
    num = lambda **k: st.column_config.NumberColumn(**k)
    ed = st.data_editor(df, key="ed_prod", hide_index=True, width="stretch", num_rows="dynamic", column_config={
        "Include": st.column_config.CheckboxColumn(default=True),
        "Zone": st.column_config.SelectboxColumn(options=ZONES, required=True, default="Apartment Development"),
        "Category": st.column_config.SelectboxColumn(options=CATEGORIES, required=True, default="Apartment"),
        "Area (sq ft)": num(min_value=0, step=1000, format="%d"), "Sales rate (₹/sq ft)": num(min_value=0, step=100, format="%d"),
        "Construction (₹/sq ft)": num(min_value=0, step=50, format="%d"),
        "Launch Q (fixed)": num(min_value=1, max_value=HORIZON_Q, step=1), "Sales period (qtrs)": num(min_value=1, max_value=HORIZON_Q, step=1),
        "Sales curve": st.column_config.SelectboxColumn(options=BASIC_CURVES, required=True, default="Even"),
        "Payment plan": st.column_config.SelectboxColumn(options=plans, required=True, default=plans[0] if plans else None),
        "Construction start Q (fixed)": num(min_value=1, max_value=HORIZON_Q, step=1),
        "Construction (qtrs)": num(min_value=1, max_value=HORIZON_Q, step=1),
        "Construction curve": st.column_config.SelectboxColumn(options=BASIC_CURVES, required=True, default="S-curve"),
        "Active": st.column_config.TextColumn(disabled=True), "Revenue (₹ Cr)": num(disabled=True, format="%.2f"),
        "Check": st.column_config.TextColumn(disabled=True, width="large")})
    if not ed.equals(df):
        old = {i: p for i, p in enumerate(P.products)}
        new = []
        for i, r in ed.reset_index(drop=True).iterrows():
            v = lambda col, d: d if pd.isna(r[col]) else r[col]
            prev = old.get(i)
            name = str(v("Product", f"New product {i+1}"))
            prod = Product(name, v("Zone", "Apartment Development"), v("Category", "Apartment"), bool(v("Include", True)),
                           float(v("Area (sq ft)", 0)), float(v("Sales rate (₹/sq ft)", 0)), float(v("Construction (₹/sq ft)", 0)),
                           int(v("Launch Q (fixed)", 1)), int(v("Sales period (qtrs)", 1)), v("Sales curve", "Even"),
                           v("Payment plan", plans[0] if plans else ""), int(v("Construction start Q (fixed)", 1)),
                           int(v("Construction (qtrs)", 1)), v("Construction curve", "S-curve"),
                           copy.deepcopy(prev.timing) if prev else Timing())
            if prev and prev.name != name:                       # renamed -> keep links pointing to it
                for q in P.products:
                    for attr in ("launch_link_product", "cons_link_product"):
                        if getattr(q.timing, attr) == prev.name: setattr(q.timing, attr, name)
                for q in new:
                    for attr in ("launch_link_product", "cons_link_product"):
                        if getattr(q.timing, attr) == prev.name: setattr(q.timing, attr, name)
            new.append(prod)
        P.products = new
        rerun_after_edit("ed_prod")
    show("Products", "names"); show("Products", "active")
    if RES:
        st.caption(f"Active saleable area {RES['costs'].total_sa:,.0f} sq ft · GDV ₹{RES['costs'].total_revenue:,.2f} Cr · "
                   f"construction ₹{RES['costs'].total_construction:,.2f} Cr")

# ============================================================================ page 4: product timing
elif page.startswith("4"):
    st.header("4 · Product timing – how each launch and construction start is decided")
    st.caption("Fixed = uses the quarter typed on page 3. Linked = X quarters after an event of a product listed ABOVE it "
               "(put the lead product first). Offset 0 = same quarter, 1 = next quarter, −1 = one before.")
    names = [p.name for p in P.products]
    notes = {n: t.note for n, t in RES["timeline"].products.items()} if RES else {}
    df = pd.DataFrame([{
        "Product": p.name, "Launch timing": p.timing.launch_mode, "Launch – linked product": p.timing.launch_link_product,
        "Launch – event": p.timing.launch_link_event, "Launch offset": p.timing.launch_offset,
        "Construction timing": p.timing.cons_mode, "Construction – linked product": p.timing.cons_link_product,
        "Construction – event": p.timing.cons_link_event, "Construction offset": p.timing.cons_offset,
        "Result": notes.get(p.name, "— (not active)"), "Check": row_status("Timing", p.name)} for p in P.products])
    sel = lambda opts: st.column_config.SelectboxColumn(options=opts)
    ed = st.data_editor(df, key="ed_time", hide_index=True, width="stretch", num_rows="fixed", column_config={
        "Product": st.column_config.TextColumn(disabled=True), "Launch timing": sel(LAUNCH_MODES),
        "Launch – linked product": sel(names), "Launch – event": sel(LINK_EVENTS),
        "Launch offset": st.column_config.NumberColumn(min_value=-8, max_value=40, step=1),
        "Construction timing": sel(CONS_MODES), "Construction – linked product": sel(names), "Construction – event": sel(LINK_EVENTS),
        "Construction offset": st.column_config.NumberColumn(min_value=-8, max_value=40, step=1),
        "Result": st.column_config.TextColumn(disabled=True, width="large"), "Check": st.column_config.TextColumn(disabled=True)})
    if not ed.equals(df):
        for p, (_, r) in zip(P.products, ed.iterrows()):
            n = lambda col, d=None: d if pd.isna(r[col]) else r[col]
            p.timing = Timing(n("Launch timing", "Fixed quarter"), n("Launch – linked product"), n("Launch – event"), int(n("Launch offset", 0)),
                              n("Construction timing", "Fixed quarter"), n("Construction – linked product"), n("Construction – event"),
                              int(n("Construction offset", 0)))
        rerun_after_edit("ed_time")
    if RES:
        for e in RES["timeline"].errors: st.error(e, icon="⚠️")
        tl = RES["timeline"]
        st.subheader("Timeline (S = selling · C = construction · # = both)")
        rows = []
        for t in tl.products.values():
            rows.append({"Product": t.name, **{f"Q{q}": ("#" if (t.launch <= q <= t.sales_end and t.cons_start <= q <= t.cons_end)
                                                      else "S" if t.launch <= q <= t.sales_end else "C" if t.cons_start <= q <= t.cons_end else "")
                                               for q in range(1, max(tl.last_q, 1) + 1)}})
        st.dataframe(pd.DataFrame(rows), hide_index=True, width="stretch")

# ============================================================================ page 5: cost heads
elif page.startswith("5"):
    st.header("5 · Cost heads")
    st.caption("Change rates, switch heads on/off and pick how each cost is spread over time. Grey 'Auto' rows are calculated "
               "elsewhere (land page, product table, calculators). Percent inputs are typed as % (4 = 4%).")
    META = {  # key: (how it is calculated, unit of input 1, unit of input 2)
        "land": ("JV: landowner share × revenue | Outright: price × area", "Auto", ""), "sd": ("From land page", "Auto", ""),
        "brok": ("% × land value", "%", ""), "ovh": ("% × total revenue", "%", ""),
        "mkt": ("Input 1 % × built-up revenue + Input 2 % × plotted revenue", "%", "%"),
        "legal": ("₹/sq ft × total saleable area", "₹/sq ft", ""), "ews": ("Loss ₹/sq ft × EWS share × saleable area", "₹/sq ft", "%"),
        "cont": ("% × base chosen in 'Basis'", "%", ""), "edc": ("Input 1 per 'Basis'", "per basis", ""),
        "hotel": ("Fee per key (₹) × number of keys", "₹/key", "keys"), "furn": ("₹/sq ft × studio area", "₹/sq ft", ""),
        "gst": ("(Commercial + plot revenue) × unsold % × GST % × developer share", "% GST", "% unsold"),
        "aif": ("Calculator page 6", "Auto", ""), "bar": ("Calculator page 6", "Auto", "")}
    for k, n, cat in CONSTRUCTION_HEADS: META[k] = (f"Σ {cat} products (page 3)", "Auto", "")
    PCT1 = {"brok", "ovh", "mkt", "cont", "gst"}; PCT2 = {"mkt", "ews", "gst"}
    amounts = RES["costs"].amounts if RES else {}
    tsa = RES["costs"].total_sa if RES else 0
    def shown1(c):
        if c.rate1 is None: return None
        is_pct = c.key in PCT1 or (c.key == "edc" and c.rate2 == "% of construction cost") or \
                 (c.key.startswith("spare") and c.basis in ("% of total revenue", "% of construction cost"))
        return c.rate1 * 100 if is_pct else c.rate1
    df = pd.DataFrame([{
        "Cost head": c.name, "How it is calculated": META.get(c.key, ("Spare: per 'Basis'", "per basis", ""))[0],
        "Input 1": shown1(c), "Unit 1": META.get(c.key, ("", "per basis", ""))[1],
        "Input 2": (c.rate2 * 100 if c.key in PCT2 else c.rate2) if isinstance(c.rate2, (int, float)) else None,
        "Unit 2": META.get(c.key, ("", "", ""))[2],
        "Basis": c.rate2 if c.key in ("cont", "edc") else (c.basis if c.key.startswith("spare") else None),
        "Applicable": c.applicable,
        "Phasing curve": ("Follow Collections" if P.info.deal_type == "JV" else P.land.payment_curve) if c.key == "land" else c.curve,
        "Start Q": c.start_q, "End Q": c.end_q,
        "Amount (₹ Cr)": round(amounts.get(c.key, 0.0), 2),
        "₹ / sq ft": round(amounts.get(c.key, 0.0) * CRORE / tsa, 0) if tsa else 0.0,
        "Check": row_status("Costs", c.key)} for c in P.costs])
    ed = st.data_editor(df, key="ed_cost", hide_index=True, width="stretch", num_rows="fixed", column_config={
        "Cost head": st.column_config.TextColumn(width="medium"), "How it is calculated": st.column_config.TextColumn(disabled=True),
        "Unit 1": st.column_config.TextColumn(disabled=True), "Unit 2": st.column_config.TextColumn(disabled=True),
        "Basis": st.column_config.SelectboxColumn(options=sorted(set(SPARE_BASES + EDC_BASES + CONTINGENCY_BASES))),
        "Phasing curve": st.column_config.SelectboxColumn(options=COST_CURVES),
        "Start Q": st.column_config.NumberColumn(min_value=1, max_value=HORIZON_Q, step=1),
        "End Q": st.column_config.NumberColumn(min_value=1, max_value=HORIZON_Q, step=1),
        "Amount (₹ Cr)": st.column_config.NumberColumn(disabled=True, format="%.2f"),
        "₹ / sq ft": st.column_config.NumberColumn(disabled=True, format="%d"),
        "Check": st.column_config.TextColumn(disabled=True, width="large")})
    if not ed.equals(df):
        auto = {"land", "sd", "aif", "bar"} | {k for k, _, _ in CONSTRUCTION_HEADS}
        for c, (_, r) in zip(P.costs, ed.iterrows()):
            g = lambda col: None if pd.isna(r[col]) else r[col]
            if c.key.startswith("spare"): c.name = str(g("Cost head") or c.name); c.basis = g("Basis")
            c.applicable = bool(r["Applicable"])
            if c.key != "land":
                c.curve = g("Phasing curve") or c.curve
                c.start_q = None if g("Start Q") is None else int(g("Start Q"))
                c.end_q = None if g("End Q") is None else int(g("End Q"))
            if c.key in auto: continue
            v1 = g("Input 1")
            if v1 is not None:
                is_pct = c.key in PCT1 or (c.key == "edc" and (g("Basis") or c.rate2) == "% of construction cost") or \
                         (c.key.startswith("spare") and c.basis in ("% of total revenue", "% of construction cost"))
                c.rate1 = float(v1) / 100 if is_pct else float(v1)
            if c.key in ("cont", "edc"):
                c.rate2 = g("Basis") or c.rate2
            elif c.key in PCT2 or c.key == "hotel":
                v2 = g("Input 2")
                c.rate2 = (float(v2) / 100 if c.key in PCT2 else float(v2)) if v2 is not None else c.rate2
        rerun_after_edit("ed_cost")
    if st.button("➕ Add a spare cost head"):
        n = sum(1 for c in P.costs if c.key.startswith("spare")) + 1
        P.costs.append(CostHead(f"spare{n}", f"Other cost {n}", rate1=0.0, applicable=True, curve="Follow Construction",
                                basis="Lump sum (Rs Cr)"))
        rerun_after_edit("ed_cost")
    if RES:
        st.metric("TOTAL COST", f"₹{RES['costs'].total_cost:,.2f} Cr", f"{RES['costs'].total_cost / RES['costs'].total_revenue:.1%} of GDV",
                  delta_color="off")

# ============================================================================ page 6: calculators
elif page.startswith("6"):
    st.header("6 · Calculators")
    t1, t2, t3 = st.tabs(["BAR / premium FAR", "AIF cost", "Initial investment"])
    with t1:
        b = P.bar
        c1, c2 = st.columns(2)
        b.authority = c1.selectbox("Authority", BAR_AUTHORITIES, BAR_AUTHORITIES.index(b.authority))
        b.land_use = c2.selectbox("Land use", BAR_LAND_USES, BAR_LAND_USES.index(b.land_use))
        b.free_far = c1.number_input("Free FAR (× land area)", value=float(b.free_far), min_value=0.0, step=0.1)
        b.loading_factor = c2.number_input("Loading factor (saleable ÷ built-up)", value=float(b.loading_factor), min_value=0.01, step=0.05)
        b.reserved_price_sqm = c1.number_input("Reserved price (₹ / sq m)", value=float(b.reserved_price_sqm), min_value=0.0, step=100.0)
        b.gst_applicable = c2.checkbox("GST on BAR applicable", b.gst_applicable)
        b.gst_rate = pct_in("GST rate on BAR (%)", b.gst_rate)
        st.write("**BAR reserved-price % table** (blank = combination not allowed)")
        df = pd.DataFrame({a: {u: (None if b.pct_table[u].get(a) is None else b.pct_table[u][a] * 100) for u in BAR_LAND_USES}
                           for a in BAR_AUTHORITIES})
        ed = st.data_editor(df, key="ed_bar")
        if not ed.equals(df):
            b.pct_table = {u: {a: (None if pd.isna(ed.loc[u, a]) else float(ed.loc[u, a]) / 100) for a in BAR_AUTHORITIES} for u in BAR_LAND_USES}
            rerun_after_edit("ed_bar")
        show("BAR", "bar")
        if RES:
            x = RES["costs"].bar
            st.success(f"Free {x['free_sqm']:,.0f} sq m · planned {x['planned_sqm']:,.0f} sq m · extra {x['extra_sqm']:,.0f} sq m × "
                       f"₹{b.reserved_price_sqm:,.0f} × {x['pct']:.0%} = **₹{x['cost']:.2f} Cr**")
    with t2:
        a = P.aif
        a.rate = pct_in("Interest rate (% p.a.)", a.rate)
        a.tenure_years = st.number_input("Tenure (years)", value=float(a.tenure_years), min_value=0.0, step=0.5)
        a.compounding = st.selectbox("Compounding", list(COMPOUNDING), list(COMPOUNDING).index(a.compounding))
        show("AIF", "aif")
        if RES:
            x = RES["costs"].aif
            st.info(f"Principal (land + stamp duty + brokerage) ₹{x['principal']:,.2f} Cr → interest **₹{x['cost']:,.2f} Cr**. "
                    f"Switch the 'AIF Cost' head on (page 5) to include it.", icon="💡")
    with t3:
        ii = P.initial
        st.write("**JV – advance to landowner and early payments** (only used in JV deals)")
        c1, c2, c3 = st.columns(3)
        ii.advance_pct = pct_in("Advance to landowner (% of land value at market)", ii.advance_pct)
        ii.dd_amount = c1.number_input("Paid on due diligence (₹ Cr)", value=float(ii.dd_amount), min_value=0.0, step=0.5)
        ii.da_amount = c2.number_input("Paid at DA (₹ Cr)", value=float(ii.da_amount), min_value=0.0, step=0.5)
        show("Initial investment", "dd_da")
        ii.legal_share = pct_in("Share of legal & compliance paid before DA (%)", ii.legal_share)
        ii.bar_share = pct_in("Share of BAR paid upfront (%)", ii.bar_share)
        ii.dev_cost_pre_da = c3.number_input("Development cost before DA (₹ Cr)", value=float(ii.dev_cost_pre_da), min_value=0.0, step=0.5)
        ii.marketing_pre_launch = c3.number_input("Pre-launch marketing (₹ Cr)", value=float(ii.marketing_pre_launch), min_value=0.0, step=0.5)
        labels = {"token": "Term-sheet token (balance)", "dd": "Due diligence", "da": "At DA", "legal": "Legal & compliance",
                  "dev": "Pre-DA development", "marketing": "Pre-launch marketing", "sd": "Stamp duty", "brok": "Brokerage", "bar": "BAR (upfront share)"}
        items = RES["costs"].initial_items if RES else {}
        df = pd.DataFrame([{"Item": labels[k], "Amount (₹ Cr)": round(items.get(k, 0.0), 2), "Payout quarter": q}
                           for k, q in ii.payout_q.items()])
        ed = st.data_editor(df, key="ed_ii", hide_index=True, num_rows="fixed", column_config={
            "Item": st.column_config.TextColumn(disabled=True), "Amount (₹ Cr)": st.column_config.NumberColumn(disabled=True, format="%.2f"),
            "Payout quarter": st.column_config.NumberColumn(min_value=1, max_value=HORIZON_Q, step=1)})
        if not ed.equals(df):
            for k, (_, r) in zip(list(ii.payout_q), ed.iterrows()):
                ii.payout_q[k] = int(r["Payout quarter"]) if not pd.isna(r["Payout quarter"]) else ii.payout_q[k]
            rerun_after_edit("ed_ii")
        show("Initial investment", "payout_q")
        st.write("**Outright – initial outlay** = land + stamp duty + brokerage + shares below")
        ii.outright_construction_share = pct_in("Share of built-up construction (%)", ii.outright_construction_share)
        ii.outright_legal_share = pct_in("Share of legal & compliance (%)", ii.outright_legal_share)
        if RES:
            cr = RES["costs"]
            st.success(f"Initial investment – JV ₹{cr.initial_jv:,.2f} Cr · Outright ₹{cr.initial_outright:,.2f} Cr · "
                       f"**used ({P.info.deal_type}) ₹{cr.initial_used:,.2f} Cr**")

# ============================================================================ page 7: payment plans
elif page.startswith("7"):
    st.header("7 · Payment plans (CLP)")
    st.caption("Booking-linked: value = quarters after booking (0–12). Construction-linked: value = % of that product's construction "
               "(e.g. 35). Each plan must total exactly 100%. Add as many milestones as you need with ➕.")
    P.catch_up_lag = int(st.number_input("Catch-up lag – late buyers pay milestones already reached this many quarters after booking",
                                         value=int(P.catch_up_lag), min_value=0, max_value=4, step=1))
    show("CLP", "catch_up_lag")
    names = [pl.name for pl in P.plans]
    used_by = {pl: [p.name for p in P.products if p.clp_plan == pl] for pl in names}
    c1, c2 = st.columns([2, 1])
    choice = c1.selectbox("Plan", names, format_func=lambda n: f"{n}  {'⚠️' if issues_for('CLP', n) else '✔'}  "
                                                              f"(used by: {', '.join(used_by[n]) or 'none'})")
    plan = P.plan(choice)
    new_name = c2.text_input("Rename plan", plan.name)
    if new_name and new_name != plan.name and new_name not in names:
        for p in P.products:
            if p.clp_plan == plan.name: p.clp_plan = new_name
        plan.name = new_name
        st.rerun()
    df = pd.DataFrame([{"Milestone": m.name, "Trigger": m.trigger,
                        "Value": m.value if m.trigger == "Booking + quarters" else round(m.value * 100, 4),
                        "% of sale": round(m.pct * 100, 4)} for m in plan.milestones])
    if df.empty:
        df = pd.DataFrame(columns=["Milestone", "Trigger", "Value", "% of sale"])
    ed = st.data_editor(df, key=f"ed_plan_{choice}", hide_index=True, width="stretch", num_rows="dynamic", column_config={
        "Trigger": st.column_config.SelectboxColumn(options=TRIGGERS, required=True, default="Construction % complete"),
        "Value": st.column_config.NumberColumn(min_value=0, max_value=100, help="quarters (booking) or % (construction)"),
        "% of sale": st.column_config.NumberColumn(min_value=0, max_value=100, step=0.5, format="%.2f")})
    if not ed.equals(df):
        ms = []
        for _, r in ed.iterrows():
            if pd.isna(r["Milestone"]) and pd.isna(r["% of sale"]): continue
            trig = r["Trigger"] if not pd.isna(r["Trigger"]) else "Construction % complete"
            val = 0.0 if pd.isna(r["Value"]) else float(r["Value"])
            ms.append(Milestone(str(r["Milestone"]) if not pd.isna(r["Milestone"]) else "Milestone", trig,
                                val if trig == "Booking + quarters" else val / 100, 0.0 if pd.isna(r["% of sale"]) else float(r["% of sale"]) / 100))
        plan.milestones = ms
        rerun_after_edit(f"ed_plan_{choice}")
    total = sum(m.pct for m in plan.milestones)
    (st.success if abs(total - 1) < 1e-6 else st.error)(f"Plan total: {total:.2%}" + ("" if abs(total - 1) < 1e-6 else " – must be exactly 100%"))
    for i in issues_for("CLP", plan.name):
        if "totals" not in i.message: st.error(i.message, icon="⚠️")
    if RES:
        sched = [s for s in RES["collections"].schedule if s.plan == plan.name and s.due_q]
        if sched:
            st.write("**When construction-linked milestones fall due** (for products using this plan)")
            st.dataframe(pd.DataFrame([{"Product": s.product, "Milestone": s.name, "Falls due": s.text} for s in sched]),
                         hide_index=True, width="stretch")
    st.divider()
    c1, c2, c3 = st.columns(3)
    nn = c1.text_input("New plan name", placeholder="e.g. CLP – 12-Floor Tower")
    if c1.button("➕ Create empty plan") and nn and nn not in names:
        P.plans.append(ClpPlan(nn, [])); st.rerun()
    if c2.button("⧉ Duplicate this plan"):
        k = 2
        while f"{plan.name} ({k})" in names: k += 1
        P.plans.append(ClpPlan(f"{plan.name} ({k})", copy.deepcopy(plan.milestones))); st.rerun()
    if c3.button("🗑 Delete this plan", disabled=bool(used_by[plan.name]),
                 help="Only plans not used by any product can be deleted."):
        P.plans = [pl for pl in P.plans if pl.name != plan.name]; st.rerun()
    show("CLP", "plan_names")

# ============================================================================ page 8: tax & benchmarks
elif page.startswith("8"):
    st.header("8 · Tax, cost of equity & KPI benchmarks")
    f = P.finance
    c1, c2, c3 = st.columns(3)
    with c1:
        f.tax_company = pct_in("Tax rate – Company (%)", f.tax_company, help="Verify with Finance (statutory 115BAA ≈ 25.17%)")
        f.tax_llp = pct_in("Tax rate – LLP (%)", f.tax_llp, help="Verify with Finance (statutory ≈ 34.94%)")
        f.cost_of_equity = pct_in("Cost of equity / discount rate (%)", f.cost_of_equity)
        show("Finance", "finance")
    labels = {"project_irr": "Project IRR (%)", "developer_irr": "Developer IRR (%)", "roce": "ROCE (%)", "cagr": "CAGR (%)",
              "pat_margin": "PAT margin (%)", "ebitda_margin": "EBITDA margin (%)"}
    with c2:
        st.write("**Benchmarks** (green when beaten)")
        for k in ["project_irr", "developer_irr", "roce"]: f.benchmarks[k] = pct_in(labels[k], f.benchmarks[k], key=f"bm_{k}")
    with c3:
        st.write("&nbsp;")
        for k in ["cagr", "pat_margin", "ebitda_margin"]: f.benchmarks[k] = pct_in(labels[k], f.benchmarks[k], key=f"bm_{k}")
        f.benchmarks["equity_multiple"] = st.number_input("Equity multiple (x)", value=float(f.benchmarks["equity_multiple"]), min_value=0.0, step=0.1)

# ============================================================================ page 9: checks & summary
else:
    st.header("9 · Checks & summary")
    if not issues and RES and not RES["problems"]:
        st.success("✔ ALL CHECKS PASSED – the results below can be used.")
    else:
        for page_name, secs in PAGE_SECTIONS.items():
            mine = [i for i in issues if i.section in secs]
            if mine:
                st.subheader(page_name)
                for i in mine: st.error(i.message, icon="⚠️")
        if RES:
            for p in RES["problems"]:
                if not any(p.endswith(i.message) for i in issues): st.error(p, icon="⚠️")
        if ENGINE_ERROR:
            st.error(f"The engine could not run with these inputs: {ENGINE_ERROR}")
        if RES is None:
            st.info("Results appear once the product, timing and payment-plan errors above are fixed.")
    if RES:
        pnl, m = RES["cashflow"].pnl, RES["cashflow"].metrics
        st.subheader("Headline results")
        c = st.columns(4)
        c[0].metric("GDV", f"₹{pnl['revenue']:,.1f} Cr"); c[1].metric("Total cost", f"₹{pnl['total_cost']:,.1f} Cr")
        c[2].metric("EBITDA", f"₹{pnl['ebitda']:,.1f} Cr", f"{pnl['ebitda_margin']:.1%} margin", delta_color="off")
        c[3].metric("PAT", f"₹{pnl['pat']:,.1f} Cr", f"{pnl['pat_margin']:.1%} margin", delta_color="off")
        c = st.columns(4)
        fmt = lambda v: "n/a" if v is None else f"{v:.1%}"
        c[0].metric("Developer IRR", fmt(m["developer_irr"])); c[1].metric("Developer NPV", f"₹{m['developer_npv']:,.1f} Cr")
        c[2].metric("Peak investment", f"₹{m['peak_investment']:,.1f} Cr", f"in Q{m['peak_quarter']}" if m["peak_quarter"] else "", delta_color="off")
        c[3].metric("Payback", f"Q{m['payback_q']}" if m["payback_q"] else "never",
                    m["payback_date"].strftime("%b %Y") if m["payback_date"] else "", delta_color="off")
        st.caption("Charts, KPI cards and takeaways are on '0 · Dashboard'.")

# ============================================================================ keep results in step with the inputs
# Fields above change the project while this page is drawn; if anything changed, redraw once so every
# number, check and error message reflects the new inputs.
if json.dumps(to_dict(P), sort_keys=True, default=str) != FINGERPRINT:
    st.rerun()


### Cell 5 – New tests (`tests/test_8_export.py`)

In [ ]:
%%writefile tests/test_8_export.py
"""STEP 9 – Excel report: builds, contains the right numbers, and can be reopened in the app."""
import io, os, sys

import numpy as np
import pytest
from openpyxl import load_workbook

from engine.export import build_excel_report, project_from_excel
from engine.inputs import to_dict
from engine.stress import monte_carlo
from conftest import variant
from engine.cashflow import run_model

SHEETS = ["Summary", "Inputs", "P&L & Metrics", "Cash Flow", "Collections", "Timeline", "Stress Test", "Monte Carlo", "_project"]


def _find(ws, label, col=1):
    for row in ws.iter_rows():
        if row[col - 1].value == label:
            return row
    raise AssertionError(f"'{label}' not found in {ws.title}")


def test_report_builds_with_all_sheets(project, base):
    wb = load_workbook(io.BytesIO(build_excel_report(project, base)))
    assert wb.sheetnames == SHEETS and wb["_project"].sheet_state == "hidden"


def test_report_numbers_match_the_engine(project, base):
    wb = load_workbook(io.BytesIO(build_excel_report(project, base)))
    pnl = _find(wb["Summary"], "PAT")
    assert abs(pnl[1].value - base["cashflow"].pnl["pat"]) < 1e-9
    dcf = _find(wb["Cash Flow"], "DEVELOPER NET CASH FLOW (pre-tax)")
    total = dcf[-1].value if dcf[-1].value is not None else [c.value for c in dcf if c.value is not None][-1]
    assert abs(total - base["cashflow"].pnl["ebitda"]) < 1e-6           # cash = profit, also in the report
    irr = _find(wb["P&L & Metrics"], "Developer IRR (pre-tax)")
    assert irr[1].value == "110.2%" and irr[3].value == "Strong"
    assert wb["Summary"]["A4"].value == "✔ ALL CHECKS PASSED"


def test_report_can_be_reopened_as_the_same_project(project, base):
    assert to_dict(project_from_excel(build_excel_report(project, base))) == to_dict(project)


def test_report_with_stress_and_monte_carlo(project, base):
    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
    pytest.importorskip("plotly")
    import stress_view
    mc = monte_carlo(project, n=25, seed=3)
    wb = load_workbook(io.BytesIO(build_excel_report(project, base, stress_view.compute_tables(project), mc)))
    best = _find(wb["Stress Test"], "Best Case")
    assert abs(best[4].value - 1.5076) < 1e-3
    runs = [r for r in wb["Monte Carlo"].iter_rows(values_only=True) if isinstance(r[0], int) and len(r) >= 8 and r[7] is not None]
    assert len(runs) == 25


def test_report_works_for_other_deals():
    for change in (lambda p: setattr(p.info, "deal_type", "Outright"), lambda p: setattr(p.info, "project_type", "Villa"),
                   lambda p: setattr(p.info, "scenario", "Worst Case")):
        p = variant(change)
        assert len(build_excel_report(p, run_model(p))) > 10_000


### Cell 6 – Safety net: run all tests (expect ✅ 170, ~20 seconds)

In [ ]:
res = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--tb=short'], capture_output=True, text=True)
print(res.stdout.strip().splitlines()[-1])
print('✅ All good – continue' if res.returncode == 0 else '❌ Something broke:\n' + res.stdout[-4000:])

### Cell 7 – Build a full report right here (with stress test and a 500-run Monte Carlo) and download it
It is also saved to Google Drive → `feasibility_app/data/`.

In [ ]:
import importlib, io
import engine.export, stress_view
importlib.reload(engine.export); importlib.reload(stress_view)
from engine.inputs import sample_project
from engine.cashflow import run_model
from engine.stress import monte_carlo
from engine.export import build_excel_report, project_from_excel
from openpyxl import load_workbook

project = sample_project()
t0 = time.time()
res = run_model(project)
stress = stress_view.compute_tables(project)
mc = monte_carlo(project, n=500, seed=42)
data = build_excel_report(project, res, stress, mc)
path = 'data/Project_Aurora_Feasibility_Report.xlsx'
open(path, 'wb').write(data)
print(f'✅ Report built in {time.time() - t0:.0f} s – {len(data) / 1024:.0f} KB – saved to {path}')
wb = load_workbook(io.BytesIO(data))
print('Sheets:', [n for n in wb.sheetnames if not n.startswith('_')])
print('\nFirst lines of the Summary sheet:')
for row in wb['Summary'].iter_rows(min_row=1, max_row=14, max_col=3, values_only=True):
    if any(row): print('  ', ' | '.join('' if v is None else str(v) for v in row))
try:
    from google.colab import files
    files.download(path)          # your browser downloads the file
except ImportError:
    pass

### Cell 8 – Proof: the report can be reopened as the same project

In [ ]:
from engine.inputs import to_dict
again = project_from_excel(open(path, 'rb').read())
print('✅ Reopened project is identical to the original' if to_dict(again) == to_dict(project) else '❌ Different!')

### Cell 9 – ▶ Start the app (Option A – Google link)

In [ ]:
subprocess.run('pkill -f "streamlit run"', shell=True); time.sleep(1)
subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true',
                  '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'],
                 cwd=BASE, stdout=open('/content/streamlit.log', 'w'), stderr=subprocess.STDOUT)
alive = False
for _ in range(30):
    time.sleep(1)
    try:
        alive = urllib.request.urlopen('http://localhost:8501/_stcore/health', timeout=2).read() == b'ok'
        if alive: break
    except Exception:
        pass
print('✅ App running' if alive else '❌ App did not start – log:\n' + open('/content/streamlit.log').read()[-2000:])
try:
    from google.colab import output
    output.serve_kernel_port_as_window(8501, anchor_text='👉 Click here to open the Feasibility app')
except ImportError:
    print('Open http://localhost:8501')

**Option B** – public link (other devices; some office networks block it).

In [ ]:
import re
subprocess.run('pkill -f cloudflared', shell=True)
CF = '/content/cloudflared'
if not os.path.exists(CF):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF); os.chmod(CF, 0o755)
subprocess.Popen([CF, 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'], stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m: url = m.group(0); break
print('Open: ' + url if url else 'No link – use Option A.')

### Cell 10 – What to try in the app
1. *(Optional)* Go to **S · Stress & Monte Carlo** and press **▶ Run Monte Carlo** – the sidebar then says *Monte Carlo: included*.
2. In the sidebar press **Build Excel report**, then **⬇️ Download Excel report**. Open the file in Excel: start with the **Summary** sheet.
3. Change any input (e.g. land area) – the sidebar says *Inputs changed since the last report – build it again*, so you never send an outdated report.
4. **📂 Open project file or Excel report** → choose the downloaded report → the app reloads exactly that project.
5. To make a PDF: open the report in Excel → *Summary* sheet → File → Save as PDF (it is set to fit one landscape page wide).

### Cell 11 – Stop the app

In [ ]:
subprocess.run('pkill -f "streamlit run"; pkill -f cloudflared', shell=True); print('Stopped.')

---
### ✅ Step 9 complete
**Next – Step 10 (final):** publish the app online with a **permanent link** (GitHub + Streamlit Community Cloud), write the GitHub README,
and prepare the LinkedIn post and screen-recording script.